# Ingenieria de caracteristicas (Feature Engineering)

Este notebook toma el dataset crudo de solicitudes de credito y aplica las transformaciones necesarias para dejarlo listo para el entrenamiento de modelos de Machine Learning.

Se carga el dataset original (`Base_de_datos.csv`) y se aplican las mismas reglas de limpieza identificadas durante el EDA (`comprension_eda.ipynb`), ademas de una nueva transformacion sobre `fecha_prestamo`.

### Transformaciones aplicadas

- **tendencia_ingresos**: se imputan los valores nulos y los valores invalidos (numericos) con la categoria "Desconocido", igual que en el EDA.
- **fecha_prestamo**: se convierte de texto a un numero entero con formato AAAAMMDD (ej: 21/12/2024 -> 20241221), para que pueda tratarse como variable numerica en vez de generar miles de columnas con OneHotEncoder.

- **Variables numericas**: se imputan los valores faltantes con la media.

- **Variables categoricas nominales** (sin orden logico, ej: tipo_laboral):
se imputan con el valor mas frecuente y se codifican con OneHotEncoder.

- **Variable categorica ordinal** (tendencia_ingresos): se imputa con el valor mas frecuente y se codifica con OrdinalEncoder, respetando el orden logico Desconocido < Decreciente < Estable < Creciente.

In [4]:
import pandas as pd

def cargarDatos():
    df = pd.read_csv('../../Base_de_datos.csv', sep=';')

    # limpieza de tendencia_ingresos (igual que en el EDA)
    df['tendencia_ingresos'] = df['tendencia_ingresos'].fillna('Desconocido')
    categorias_validas = ['Creciente', 'Decreciente', 'Estable']
    df.loc[~df['tendencia_ingresos'].isin(categorias_validas), 'tendencia_ingresos'] = 'Desconocido'

    # conversion de fecha_prestamo a numero entero formato AAAAMMDD
    df['fecha_prestamo'] = pd.to_datetime(df['fecha_prestamo'], format='%d/%m/%Y %H:%M')
    df['fecha_prestamo'] = df['fecha_prestamo'].dt.strftime('%Y%m%d').astype(int)

    return df

from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

df = cargarDatos()

print(df.head())
print(df.info())
print(df.describe())

X = df.drop('Pago_atiempo', axis=1)
y = df['Pago_atiempo']

num_features = X.select_dtypes('number').columns
ordinal_features = ['tendencia_ingresos']
cat_features = X.select_dtypes('object').columns.drop(ordinal_features)

print("Numeric features")
print(num_features)
print("Categorical (nominal) features")
print(cat_features)
print("Categorical (ordinal) features")
print(ordinal_features)

num_transformer = Pipeline(steps=[
    ('inputer', SimpleImputer(strategy='mean'))
])

cat_transformer = Pipeline(steps=[
    ('inputer', SimpleImputer(strategy='most_frequent')),
    ('to_str', FunctionTransformer(lambda x: x.astype(str))),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

orden_tendencia = ['Desconocido', 'Decreciente', 'Estable', 'Creciente']

ordinal_transformer = Pipeline(steps=[
    ('inputer', SimpleImputer(strategy='most_frequent')),
    ('to_str', FunctionTransformer(lambda x: x.astype(str))),
    ('ordinal', OrdinalEncoder(categories=[orden_tendencia]))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features),
        ('ord', ordinal_transformer, ordinal_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train preprocesados:")
print(X_train_processed)
print(X_train_processed.shape)
print("X_test preprocesados:")
print(X_test_processed)
print(X_test_processed.shape)

def ft_engineering():
    num_features = X.select_dtypes('number').columns
    ordinal_features = ['tendencia_ingresos']
    cat_features = X.select_dtypes('object').columns.drop(ordinal_features)

    num_transformer = Pipeline(steps=[
        ('inputer', SimpleImputer(strategy='mean'))
    ])

    cat_transformer = Pipeline(steps=[
        ('inputer', SimpleImputer(strategy='most_frequent')),
        ('to_str', FunctionTransformer(lambda x: x.astype(str))),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    orden_tendencia = ['Desconocido', 'Decreciente', 'Estable', 'Creciente']
    ordinal_transformer = Pipeline(steps=[
        ('inputer', SimpleImputer(strategy='most_frequent')),
        ('to_str', FunctionTransformer(lambda x: x.astype(str))),
        ('ordinal', OrdinalEncoder(categories=[orden_tendencia]))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_features),
            ('cat', cat_transformer, cat_features),
            ('ord', ordinal_transformer, ordinal_features)
        ]
    )

    return preprocessor

   tipo_credito  fecha_prestamo  capital_prestado  plazo_meses  edad_cliente  \
0             7        20241221           3692160           10            42   
1             4        20250422            840000            6            60   
2             9        20260108           5974028           10            36   
3             4        20250804           1671240            6            48   
4             9        20250426           2781636           11            44   

    tipo_laboral  salario_cliente  total_otros_prestamos  cuota_pactada  \
0  Independiente          8000000                2500000         341296   
1       Empleado          3000000                2000000         124876   
2  Independiente          4036000                 829000         529554   
3       Empleado          1524547                 498000         252420   
4       Empleado          5000000                4000000         217037   

     puntaje  ...  saldo_mora  saldo_total  saldo_principal  \
0  88

### Conclusiones del feature engineering

El dataset crudo, con 22 variables predictoras, se transformo en una matriz de 247 columnas numericas lista para el entrenamiento de modelos.

**Resultado del split:**
- X_train preprocesado: 8610 filas x 247 columnas
- X_test preprocesado: 2153 filas x 247 columnas

**Decisiones clave tomadas:**

- Se evito el uso de OneHotEncoder sobre `fecha_prestamo`, ya que al tratarse de una variable con miles de valores unicos generaba una explosion de columnas (de 23 a 8649), perjudicando la eficiencia del modelo sin aportar
valor real.

- Se trato `tendencia_ingresos` como variable ordinal en lugar de nominal, respetando la relacion logica de orden entre sus categorias y conservando los registros "Desconocido" (26.7% del dataset) como categoria propia, en
vez de descartarlos.


Con los datos ya transformados, el siguiente paso es el entrenamiento y
evaluacion de los primeros modelos supervisados.